In [ ]:
# Threshold scan display, MIDAS channel -> MuTrig channel mapping

import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


FILENAME = "../data/tthreshold_scans/tth_scan_13-10-2026-15:26.json"


with open(FILENAME, "r") as f:
    data = json.load(f)

plot_groups = ["Raw","Calibrated"]
mutrig_channels = range(5)

ncols=1 ; nrows=2
fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=list(plot_groups.keys()),
    shared_xaxes=True,
    shared_yaxes=True,
)

all_rates = []

for i, (title) in enumerate(plot_groups.items()):
    row = i // nrows + 1
    col = i % ncols + 1

    for mt in mutrig_channels:
        midas = FIRST_MIDAS_CHANNEL + mt

        if midas >= len(data["th"]) or midas >= len(data["rates"]):
            continue
        if len(data["th"][midas]) == 0 or len(data["rates"][midas]) == 0:
            continue

        th = np.array(data["th"][midas])
        rates = np.array(data["rates"][midas])
        all_rates.extend(rates[rates > 0])

        fig.add_trace(
            go.Scatter(
                x=th,
                y=rates,
                mode="lines+markers",
                name=f"MuTrig {mt:02d} / MIDAS {midas}",
                hovertemplate=(
                    f"MuTrig {mt:02d}<br>"
                    f"MIDAS {midas}<br>"
                    "Threshold DAC: %{x}<br>"
                    "Rate: %{y:.3g}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

yrange = [-1, 5]
if len(all_rates):
    yrange = [
        np.log10(min(all_rates)) - 0.25,
        np.log10(max(all_rates)) + 0.25,
    ]

fig.update_xaxes(
    title_text="Threshold DAC",
    range=[0, 255],
    tickmode="linear",
    dtick=10,
)

fig.update_yaxes(
    title_text="Rates",
    type="log",
    range=yrange,
)

fig.update_layout(
    title=(
        "Threshold scan by connector"
        if PLOT_BY_CON else
        "Threshold scan by MuTrig channel group"
    ),
    height=900,
    width=1400,
    template="plotly_white",
    hovermode="closest",
    legend=dict(
        groupclick="toggleitem",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
)

fig.show()